In [1]:
import numpy as np
import pandas as pd
import os
import scipy.io
from tensorflow import keras
from keras.utils import load_img,np_utils, img_to_array, to_categorical
from sklearn.model_selection import train_test_split
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
#from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
image_folder = 'flowers/'
label_file = 'imagelabels.mat'

In [3]:
labels = scipy.io.loadmat(label_file)
data = labels['labels']

In [4]:
num_classes = np.max(data) + 1
data = to_categorical(data, num_classes=num_classes)


In [5]:
data = data.reshape(data.shape[1:])
data.shape

(8189, 103)

In [6]:
image_files = os.listdir(image_folder)
print(len(image_files))
files = [item for item in image_files if ')' not in item]
print(len(files))
files1 = [item for item in files if 'ipynb_checkpoints' not in item]
print(len(files1))

8189
8189
8189


In [7]:
image_files = os.listdir(image_folder)
print(len(files1))
images = []
for filename in files1:
  img = load_img(image_folder+"/"+filename, target_size=(128, 128))  # VGG16 expects input of size 224x224
  img_array = img_to_array(img)
  images1 = np.expand_dims(img_array, axis=0)
  images.append(images1)
images = np.array(images)
print(images.shape)

8189
(8189, 1, 128, 128, 3)


In [8]:
images /= 255.
images = images.reshape(8189, 128, 128, 3)
images.shape

(8189, 128, 128, 3)

In [9]:
train_images, test_images, train_labels, test_labels = train_test_split(images, data,train_size=0.8 ,test_size=0.2, random_state=42)

In [10]:
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

(6551, 128, 128, 3)
(6551, 103)
(1638, 128, 128, 3)
(1638, 103)


In [11]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

for layer in base_model.layers:
    layer.trainable = False

x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [14]:
model = Model(inputs=base_model.input, outputs=predictions)

In [15]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [16]:
model.fit(train_images, train_labels, epochs=10, validation_data=(test_images, test_labels))

Epoch 1/10
205/205 [==============================] - 449s 2s/step - loss: 4.4735 - accuracy: 0.0524 - val_loss: 4.1156 - val_accuracy: 0.1777
Epoch 2/10
205/205 [==============================] - 314s 2s/step - loss: 3.9420 - accuracy: 0.1465 - val_loss: 3.5541 - val_accuracy: 0.2662
Epoch 3/10
205/205 [==============================] - 417s 2s/step - loss: 3.4507 - accuracy: 0.2442 - val_loss: 3.1250 - val_accuracy: 0.3681
Epoch 4/10
205/205 [==============================] - 403s 2s/step - loss: 3.0709 - accuracy: 0.3064 - val_loss: 2.7855 - val_accuracy: 0.4389
Epoch 5/10
205/205 [==============================] - 436s 2s/step - loss: 2.7890 - accuracy: 0.3630 - val_loss: 2.5540 - val_accuracy: 0.4866
Epoch 6/10
205/205 [==============================] - 815s 4s/step - loss: 2.5435 - accuracy: 0.4123 - val_loss: 2.3463 - val_accuracy: 0.5183
Epoch 7/10
205/205 [==============================] - 964s 5s/step - loss: 2.3648 - accuracy: 0.4451 - val_loss: 2.1835 - val_accuracy: 0.5440

In [17]:
loss, accuracy = model.evaluate(test_images, test_labels)
print(f"Test Accuracy: {accuracy * 100}%")

52/52 [==============================] - 58s 1s/step - loss: 1.8470 - accuracy: 0.6050
Test Accuracy: 60.50060987472534%
